In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Saving_pi_pi_interactions)=
# Saving aromatic interaction analyses

*Calculating, naming, persisting and querying sparse aromatic ring observations.*

Keep a geometric analysis beside the system that supplied its indices. This
controlled two-ring ensemble has parallel, absent and edge-to-face observations.
The cutoffs define an experimental geometric criterion, not an energy model.
The SMILES conversion requires the optional RDKit dependency.

:::{versionadded} 1.0.0
:::

## Preparing the chemical system

Create declared aromatic chemistry and a controlled coordinate ensemble.

In [2]:
import molsysmt as msm
import numpy as np
from molsysmt import pyunitwizard as puw

# RDKit supplies explicit aromatic atom/bond flags through this form conversion.
topology = msm.convert('smiles:c1ccccc1.c1ccccc1', to_form='molsysmt.Topology')
molsys = msm.convert(topology, to_form='molsysmt.MolSys')
phase = np.arange(6) * np.pi / 3
ring = .14 * np.column_stack((np.cos(phase), np.sin(phase), np.zeros(6)))
coordinates = np.array([
    np.concatenate((ring, ring + [0, 0, .35])),
    np.concatenate((ring, ring + [0, 0, 2.])),
    np.concatenate((ring, ring[:, [2, 1, 0]] + [0, 0, .4])),
])
molsys.structures.append(coordinates=puw.quantity(coordinates, 'nm'))

def calculate(source=molsys, **options):
    settings = dict(distance_threshold='.6 nm', angle_threshold='30 degrees',
                    offset_threshold='.2 nm', planarity_threshold='.02 nm', pbc=False)
    settings.update(options)
    return msm.interactions.pi_pi.get_pi_pi_interactions(source, **settings)

analysis = calculate()
print(analysis.n_interactions, analysis.occurrence_structures.tolist())
print(analysis.measure_units)
assert analysis.n_interactions == 2
assert not molsys.interactions

2 [0, 2]
{'distance': 'nm', 'plane_angle': 'radians', 'offset_a': 'nm', 'offset_b': 'nm', 'rms_deviation_a': 'nm', 'rms_deviation_b': 'nm', 'max_deviation_a': 'nm', 'max_deviation_b': 'nm'}


## Saving the named analysis

Attachment declares correspondence with the system atom and structure axes.
H5MSM 0.5 preserves both coordinates and the named sparse analysis.

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

molsys.interactions = {**molsys.interactions, 'aromatic_contacts': analysis}
with TemporaryDirectory() as directory:
    filename = str(Path(directory) / 'aromatic.h5msm')
    msm.convert(molsys, to_form=filename)
    loaded = msm.convert(filename, to_form='molsysmt.MolSys')
    saved = loaded.interactions['aromatic_contacts']
    with msm.configure.context(chunk_size=1):
        from_file = calculate(source=filename, heavy_mode='force', structure_indices=[2, 0, 2, 1])
    np.testing.assert_array_equal(saved.query().to_dict()['occurrence_indices'],
                                  analysis.query().to_dict()['occurrence_indices'])
    np.testing.assert_array_equal(from_file.occurrence_structures, analysis.occurrence_structures)
    for name in analysis.measurements:
        np.testing.assert_allclose(saved.measurements[name], analysis.measurements[name])
        np.testing.assert_allclose(from_file.measurements[name], analysis.measurements[name])
    assert saved.parameters == analysis.parameters
    assert saved.software == analysis.software
print(saved.software)
print(saved.evaluated_structure_indices.tolist())

{'molsysmt': '0.21.0+606.ga03eb4bf6', 'networkx': '3.6.1'}
[0, 1, 2]


## Querying the visible structure

An evaluated empty frame is not an unevaluated frame. Queries preserve occurrence
indices for selection and inspection by a client such as MolSysViewer.

In [4]:
visible = saved.query(structure_indices=[2], atom_indices=[0], mode='incident').to_dict()
empty = saved.query(structure_indices=[1]).to_dict()
assert len(visible['occurrence_indices']) == 1
assert empty['evaluated_structure_indices'].tolist() == [1]
assert len(empty['occurrence_indices']) == 0
print('Visible occurrence indices:', visible['occurrence_indices'])

Visible occurrence indices: [1]


## Saving a standalone layer

A standalone analysis needs an explicit declaration of correspondence before
attachment to another system; matching axis sizes alone does not prove identity.

In [5]:
with TemporaryDirectory() as directory:
    filename = str(Path(directory) / 'analysis_only.h5msm')
    msm.h5msm.write_layers(filename, interactions={'aromatic_contacts': analysis})
    layers = msm.h5msm.read_layers(filename, layers='interactions')
    standalone = layers['interactions']['aromatic_contacts']
assert standalone.software == analysis.software
assert standalone.n_interactions == analysis.n_interactions
print('Sparse numeric bytes:', standalone.numeric_nbytes)

Sparse numeric bytes: 656


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Getting pi-pi interactions <Tutorial_Get_pi_pi_interactions>` — full criterion
  and selection modes, {func}`molsysmt.interactions.pi_pi.get_pi_pi_interactions`.
- {ref}`user-foundations-native-world-classes-molsysmt-molsys` — domain ownership and
  explicit named attachment.
:::